# Outline Artifact Viewer (By Run Timestamp)

This notebook prints outline artifacts for a specific run timestamp, grouped by attempts, expansions, revisions, preliminary plan drafts/refinements, and the plan. Each section includes the exact system prompt and the user prompt template used by the LLM for that step, with placeholders where run-specific data is injected.

Prompt blocks can be hidden by setting `SHOW_PROMPTS = False` in the switch cell.


In [2]:
from pathlib import Path
import json
import re

ARTIFACTS_DIR = Path("artifacts")

# Optional: set the run timestamp you want to inspect (from the filename prefix).
# Example: "2026_02_25_02_07_05"
RUN_TIMESTAMP = "2026_02_25_22_59_27"

# Optional: set RUN_ID if multiple runs share the same timestamp.
RUN_ID = None  # e.g., 6

# If RUN_TIMESTAMP is None, pick the most recent outline artifact automatically.
AUTO_PICK_LATEST = True


def _run_prefix_from_path(path: Path) -> str | None:
    match = re.match(
        r"^(\d{4}_\d{2}_\d{2}_\d{2}_\d{2}_\d{2})_outline_run_(\d+)_outline_",
        path.name,
    )
    if not match:
        return None
    return f"{match.group(1)}_outline_run_{match.group(2)}"


def _latest_outline_artifact() -> Path | None:
    patterns = [
        "*_outline_run_*_outline_revision*.json",
        "*_outline_run_*_outline_expand*.json",
        "*_outline_run_*_outline_attempt*.json",
    ]
    candidates = []
    for pattern in patterns:
        candidates.extend(ARTIFACTS_DIR.glob(pattern))
    if not candidates:
        return None
    return max(candidates, key=lambda p: p.stat().st_mtime)


def _pick_run_prefix():
    if RUN_ID is not None and RUN_TIMESTAMP:
        return f"{RUN_TIMESTAMP}_outline_run_{RUN_ID}"

    if not RUN_TIMESTAMP and AUTO_PICK_LATEST:
        latest = _latest_outline_artifact()
        if latest:
            return _run_prefix_from_path(latest)

    if not RUN_TIMESTAMP:
        return None

    candidates = list(ARTIFACTS_DIR.glob(f"{RUN_TIMESTAMP}_outline_run_*_outline_*.json"))
    prefixes = sorted({_run_prefix_from_path(p) for p in candidates} - {None})
    if len(prefixes) == 1:
        return prefixes[0]
    if len(prefixes) > 1:
        latest = max(candidates, key=lambda p: p.stat().st_mtime)
        return _run_prefix_from_path(latest)
    return None


def _print_outline_json(path: Path | None, label: str):
    print(f"=== {label} ===")
    if not path:
        print("(not found)")
        return None
    print(f"File: {path}")
    data = json.loads(path.read_text(encoding="utf-8"))
    outline = data.get("outline")
    if isinstance(outline, list):
        for idx, item in enumerate(outline, 1):
            print(f"{idx}. {item}")
    else:
        print(json.dumps(data, indent=2, ensure_ascii=False))
    return outline if isinstance(outline, list) else None


def _print_outline_group(paths, label):
    print(f"=== {label} ===")
    if not paths:
        print("(not found)")
        return
    for path in paths:
        _print_outline_json(path, path.name)


def _print_markdown(path: Path | None, label: str):
    print(f"=== {label} ===")
    if not path:
        print("(not found)")
        return
    print(f"File: {path}")
    print(path.read_text(encoding="utf-8"))


RUN_PREFIX = _pick_run_prefix()
print(f"RUN_PREFIX: {RUN_PREFIX}")



def _plan_run_prefix_from_path(path: Path) -> str | None:
    match = re.match(
        r"^(\d{4}_\d{2}_\d{2}_\d{2}_\d{2}_\d{2})_plan_run_(\d+)_plan_",
        path.name,
    )
    if not match:
        return None
    return f"{match.group(1)}_plan_run_{match.group(2)}"


def _latest_plan_artifact() -> Path | None:
    patterns = [
        "*_plan_run_*_plan_draft_round*.json",
        "*_plan_run_*_plan_refine_round*.json",
        "*_plan_draft_round*.json",
        "*_plan_refine_round*.json",
    ]
    candidates = []
    for pattern in patterns:
        candidates.extend(ARTIFACTS_DIR.glob(pattern))
    if not candidates:
        return None
    return max(candidates, key=lambda p: p.stat().st_mtime)


def _pick_plan_prefix(outline_prefix: str | None = None):
    if RUN_ID is not None and RUN_TIMESTAMP:
        return f"{RUN_TIMESTAMP}_plan_run_{RUN_ID}"

    if outline_prefix:
        match = re.match(r"^(\d{4}_\d{2}_\d{2}_\d{2}_\d{2}_\d{2})_outline_run_(\d+)", outline_prefix)
        if match:
            return f"{match.group(1)}_plan_run_{match.group(2)}"

    if not RUN_TIMESTAMP and AUTO_PICK_LATEST:
        latest = _latest_plan_artifact()
        if latest:
            return _plan_run_prefix_from_path(latest)

    if not RUN_TIMESTAMP:
        return None

    candidates = list(ARTIFACTS_DIR.glob(f"{RUN_TIMESTAMP}_plan_run_*_plan_*.json"))
    prefixes = sorted({_plan_run_prefix_from_path(p) for p in candidates} - {None})
    if len(prefixes) == 1:
        return prefixes[0]
    if len(prefixes) > 1:
        latest = max(candidates, key=lambda p: p.stat().st_mtime)
        return _plan_run_prefix_from_path(latest)
    return None


def _print_plan_json(path: Path | None, label: str):
    print(f"=== {label} ===")
    if not path:
        print("(not found)")
        return None
    print(f"File: {path}")
    data = json.loads(path.read_text(encoding="utf-8"))
    print(json.dumps(data, indent=2, ensure_ascii=False))
    return data


def _print_plan_group(paths, label):
    print(f"=== {label} ===")
    if not paths:
        print("(not found)")
        return
    for path in paths:
        _print_plan_json(path, path.name)


RUN_PREFIX: 2026_02_25_22_59_27_outline_run_5


In [ ]:
from IPython.display import Markdown, display

# Switch: set to False to suppress prompt display.
SHOW_PROMPTS = True

def display_prompt(md: str):
    if SHOW_PROMPTS:
        display(Markdown(md))


## Outline Attempts

Input is the topic, methods, selected documents, and KG fact cards assembled during the run. The outliner triggers attempts when it first tries to generate a compliant outline; each attempt is a fresh LLM call saved immediately after parsing. Attempts occur before any length-expansion or revision logic is applied.


In [ ]:
display_prompt("""
System prompt used by the LLM:
```text
You are a research planning agent. Produce a research plan (not a report outline). All natural-language content MUST be in the requested output language.
Return ONE valid JSON object only. No markdown, no commentary, no extra text.

Requirements:
- Must include key: outline
- outline MUST be a non-empty array (8-12 major steps).
- Each major step must include 3-5 substeps.
- Each substep should be 2-3 sentences.
- Optional sub-substeps are allowed when helpful.
- Use imperative phrasing (e.g., "Search...", "Compare...", "Investigate...").
- Do not write report section headings.
- Total outline length MUST be 1000-2000 words across titles and substeps in the output language.
- Output must be strict JSON (double quotes, no trailing commas)
- Treat methods as analysis approaches to apply to the topic, not as the topic itself.
- If methods are provided, the research plan MUST explicitly structure steps around those methods.
- All natural-language content MUST be in the requested output language.
- Keep JSON keys in English.
- Keep paper titles, dataset names, benchmarks, model names, APIs, and acronyms in English.
- Use ASCII punctuation for JSON (":" and ","), not full-width punctuation.
- When evidence includes confidence or recency, prefer higher-confidence and more recent evidence.
All natural-language content MUST be in the requested output language.

Example (Chinese):
{"outline":[{"title":"查找官方技术报告与发布博客以提取架构与训练细节。","substeps":["列出要检索的官方来源与发布页面。","提取训练规模、数据来源与能力摘要。",{"text":"捕捉部署背景与版本历史。","substeps":["记录发布节奏与更新日志。","整理公开的限制与注意事项。"]}]},{"title":"比较不同模型的上下文长度与长上下文准确性。","substeps":["收集厂商声明与独立基准。","分析长上下文召回与检索性能。",{"text":"识别失效案例。","substeps":["总结常见错误模式。","记录实践中的缓解策略。"]}]}]}

Example (English):
{"outline":[{"title":"Search for official technical reports and release blogs to extract architecture and training details.","substeps":["List official sources and release pages to target.","Extract training scale, data sources, and capability summaries.",{"text":"Capture deployment context and version history.","substeps":["Note release cadence and changelogs.","Record published caveats or limitations."]}]},{"title":"Compare context window sizes and long-context accuracy across models.","substeps":["Collect vendor claims and independent benchmarks.","Analyze long-context recall and retrieval performance.",{"text":"Identify failure cases.","substeps":["Summarize common error patterns.","Note mitigation strategies reported by practitioners."]}]}]}
```

User prompt template sent to the LLM:
```json
{
  "topic": "<topic>",
  "interests": ["<interest>", "..."],
  "methods": ["<method>", "..."],
  "documents": [
    {"title": "<title>", "source_type": "<type>", "source": "<source>", "doc_id": "<id>"}
  ],
  "keywords": ["<keyword>", "..."],
  "output_language": "<language>",
  "instructions": {
    "output_json_schema": {"outline": ["string"]},
    "language_policy": [
      "All natural-language content MUST be in <language>.",
      "Keep paper titles, dataset names, benchmarks, model names, APIs, and acronyms in English.",
      "All natural-language content MUST be in <language>."
    ],
    "outline_expectations": [
      "Include major sections and subtopics to expand later.",
      "Provide 8-12 major steps.",
      "Each major step must include 3-5 substeps.",
      "Each substep should be 2-3 sentences.",
      "List source-backed bullets when available.",
      "Emphasize gaps and future research directions.",
      "Length must be 1000-2000 words in the output language."
    ],
    "method_guidance": [
      "Use methods as analytical lenses applied to the topic and evidence.",
      "Do not treat methods as the topic itself.",
      "If methods are provided, align major steps to the methods and weave method names into the step text.",
      "Do not add bracketed method tags in titles.",
      "If method names are not in <language>, translate them into <language> step titles and include the original in parentheses."
    ],
    "kg_guidance": [
      "If kg_fact_cards is provided, use it to ground claims and outline steps.",
      "Prefer higher-confidence and more recent evidence when shaping the outline.",
      "Prefer evidence-backed relationships over unsupported assertions.",
      "Do not copy evidence snippets verbatim; paraphrase.",
      "When you use kg_fact_cards, add a short citation in-line like 'Sources: title1; title2'.",
      "Do not fabricate sources that are not in kg_fact_cards or documents.",
      "Use trend stats and contradictions in kg_fact_cards to highlight agreements, shifts, and disagreements.",
      "Dates in kg_fact_cards come from document added_at timestamps (ingestion time), not publication dates."
    ]
  },
  "kg_fact_cards": ["<fact card>", "..."]
}
```
""")


In [3]:
if RUN_PREFIX:
    attempts = sorted(ARTIFACTS_DIR.glob(RUN_PREFIX + "_outline_attempt*.json"))
else:
    attempts = []
_print_outline_group(attempts, "All Outline Attempts")

=== All Outline Attempts ===
=== 2026_02_25_22_59_27_outline_run_5_outline_attempt1.json ===
File: artifacts/2026_02_25_22_59_27_outline_run_5_outline_attempt1.json
1. Systems Engineering: 运用趋势分析 (Trend Analysis) 追溯AI智能体工作流的演进脉络与发展动力
- 梳理从传统工作流自动化到AI智能体驱动范式的演变历程。调查早期专家系统,基于规则的自动化工具,并将其与现代基于LLM的智能体进行对比,明确技术发展的关键拐点和驱动因素,为理解当前的技术格局提供历史视角。
- 分析大型语言模型(LLM)作为催化剂的核心作用。深入研究LLM的出现如何从根本上改变了智能体的能力边界,使其具备了前所未有的自然语言理解,推理和规划能力,从而催生了“代理科学”(Agentic Science)等新概念。来源: From AI for Science to Agentic Science: A Survey on Autonomous Scientific Discovery
- 识别并记录AI智能体发展史上的里程碑式项目和框架。系统性地收集从早期多智能体系统(Multi-Agent Systems)研究到当前如Auto-GPT,BabyAGI等开源项目的演进数据,绘制技术成熟度曲线和社区关注度趋势图,以量化该领域的增长和演变。
- 预测未来几年AI智能体驱动工作流的关键发展趋势。综合分析学术界和工业界的最新研究成果与产品发布,预测在自主性,多模态能力,人机协作以及垂直领域深度集成等方面可能出现的突破性进展,为未来的技术路线图规划提供依据。
2. Systems Engineering: 采用系统分析 (Systems Analysis) 方法论,明确AI智能体工作流的需求,边界与约束
- 定义AI智能体工作流系统的功能性与非功能性需求。通过分析典型应用场景(如科学研究,商业流程自动化),提炼出对任务分解,工具使用,错误处理,可解释性,安全性等方面的具体需求,并将其结构化为一份正式的系统需求规约(SyRS)。
- 划定系统的边界和上下文环境。明确智能体工作流的输入(目标,数据),输出

## Outline Expansions

Input is the best available outline that is too short. Expansion is triggered when the outline word count falls below the minimum length; the model is asked to add detail without removing content, using a length hint derived from the previous outline. Expansion runs after attempts fail length validation and before revision is enforced.


In [ ]:
display_prompt("""
System prompt used by the LLM:
```text
You expand research plan outlines to meet strict minimum length. Return JSON only.
```

User prompt template sent to the LLM:
```json
{
  "previous_outline": ["<outline step>", "..."],
  "instructions": {
    "output_json_schema": {"outline": ["string"]},
    "requirements": [
      "Length must be at least 1000 words in the output language.",
      "Keep 8-12 major steps.",
      "Each major step must include 3-5 substeps.",
      "Each substep should be 2-3 sentences.",
      "Do not remove content; only expand with additional detail and substeps."
    ],
    "language_guidance": [
      "Write natural-language content in <language>.",
      "Keep paper titles, dataset names, benchmarks, model names, APIs, and acronyms in English."
    ],
    "structure_guidance": ["<optional guidance>"]
  },
  "length_hint": "<length hint>",
  "language_hint": "<optional language hint>"
}
```
""")


In [4]:
if RUN_PREFIX:
    expands = sorted(ARTIFACTS_DIR.glob(RUN_PREFIX + "_outline_expand*.json"))
else:
    expands = []
_print_outline_group(expands, "All Outline Expansions")

=== All Outline Expansions ===
=== 2026_02_25_22_59_27_outline_run_5_outline_expand4.json ===
File: artifacts/2026_02_25_22_59_27_outline_run_5_outline_expand4.json
1. 系统工程:执行趋势分析 (Trend analysis) 以描绘 AI 智能体工作流的演进轨迹
- 追溯 AI 智能体技术的历史发展路径。本研究将深入调查从早期基于规则和逻辑的符号 AI(例如早期的专家系统和规划器)以及经典的分布式人工智能中的多智能体系统(Multi-Agent Systems, MAS)开始的演化历程。我们将详细对比这些早期范式中智能体确定性,可解释但扩展性受限的特点,与当前由大型语言模型(LLM)驱动的现代范式所展现出的概率性,涌现式和高度灵活的行为模式,从而明确各个发展阶段的核心设计思想,技术突破及其固有的局限性,如符号 AI 的知识获取瓶颈和早期 MAS 的复杂协调难题。
- 识别并记录推动智能体发展的关键技术拐点。此部分将重点分析 Transformer 架构的出现为何是革命性的,不仅因为它通过自注意力机制有效解决了长距离依赖问题,更关键的是其高度并行化的计算特性为训练万亿参数级别的大模型奠定了基础。我们还将深入探讨大规模预训练范式的成功如何将 AI 开发从构建特定任务模型转变为适应通用基础模型,以及像 Hugging Face 这样的开源社区和平台如何在模型,数据集和工具的普及化方面发挥了核心作用,极大地加速了全球范围内的技术迭代和创新民主化进程。
- 按时间顺序梳理具有影响力的智能体框架和概念的出现。我们将系统性地记录关键概念的提出和演进,例如“思维链”(Chain-of-Thought, CoT)如何通过显式引导模型进行逐步推理,显著提升了其在复杂问题上的解决能力。在此基础上,我们将剖析 ReAct (Reason+Act) 框架如何巧妙地将推理(思想)与行动(工具使用)交织在一起,形成了一个动态的,与外部世界接地的执行循环,更有效地模拟了人类解决问题的过程。此外,我们还将考察如“思维树”(Tree of Thoughts)等更复杂的规划与推理策略,分析它们如何进一步增强智能体在面对不确定性和多路径选择

## Outline Revisions

Input is the last outline that still violates length or language constraints. Revision is triggered after attempts and expansions fail to meet the strict word-count target; the model is instructed to rewrite while preserving structure, using a length hint based on the previous outline. Revision occurs near the end of the outline pipeline.


In [ ]:
display_prompt("""
System prompt used by the LLM:
```text
You revise research plan outlines to meet strict length constraints. Return JSON only.
```

User prompt template sent to the LLM:
```json
{
  "previous_outline": ["<outline step>", "..."],
  "instructions": {
    "output_json_schema": {"outline": ["string"]},
    "requirements": [
      "Length must be 1000-2000 words in the output language.",
      "Keep 8-12 major steps.",
      "Each major step must include 3-5 substeps.",
      "Each substep should be 2-3 sentences.",
      "Preserve the original topic coverage and structure; rewrite to fit length."
    ],
    "language_guidance": [
      "Write natural-language content in <language>.",
      "Keep paper titles, dataset names, benchmarks, model names, APIs, and acronyms in English."
    ],
    "structure_guidance": ["<optional guidance>"]
  },
  "length_hint": "<length hint>",
  "language_hint": "<optional language hint>"
}
```
""")


In [5]:
if RUN_PREFIX:
    revisions = sorted(ARTIFACTS_DIR.glob(RUN_PREFIX + "_outline_revision*.json"))
else:
    revisions = []
_print_outline_group(revisions, "All Outline Revisions")

=== All Outline Revisions ===
=== 2026_02_25_22_59_27_outline_run_5_outline_revision6.json ===
File: artifacts/2026_02_25_22_59_27_outline_run_5_outline_revision6.json
1. 系统工程: 执行趋势分析以描绘AI智能体工作流的演进轨迹
- 追溯AI智能体技术的历史发展路径,对比不同范式。本研究将深入考察智能体从早期基于规则和逻辑的符号AI(如专家系统)及经典分布式AI中的多智能体系统(MAS)的演化历程。我们将详细分析这些早期范式中行为的确定性和可解释性,并指出其在知识获取和扩展性方面的局限性;随后,将其与当前由大型语言模型(LLM)驱动的现代范式进行对比,后者展现出概率性,涌现式和高度灵活的行为模式,从而明确各阶段的核心设计思想与技术突破。
- 识别并记录推动智能体发展的关键技术拐点。此部分将重点分析Transformer架构为何是革命性的,它不仅通过自注意力机制有效解决了长距离依赖问题,其高度并行化的计算特性更为训练万亿参数级别的大模型奠定了基础。我们还将深入探讨大规模预训练范式如何将AI开发从构建特定任务模型转变为适应通用基础模型,以及Hugging Face等开源社区和平台如何在模型,数据集和工具的普及化方面发挥了核心作用,极大地加速了全球范围内的技术迭代和创新民主化进程。
- 按时间顺序梳理具有影响力的智能体框架与概念的出现。我们将系统性地记录关键概念的演进,例如“思维链”(Chain-of-Thought, CoT)如何通过显式引导模型进行逐步推理,显著提升了其解决复杂问题的能力。在此基础上,我们将剖析ReAct框架如何巧妙地将推理(思想)与行动(工具使用)交织在一起,形成一个与外部世界接地的动态执行循环。此外,我们还将考察如“思维树”(Tree of Thoughts)等更复杂的规划与推理策略,分析它们如何进一步增强智能体在面对不确定性和多路径选择时的决策质量与探索广度。
- 剖析“智能体”概念本身的演变与范畴扩展。本研究将探讨该术语如何从最初在分布式计算领域中定义的,具备自主性,社会性,反应性和主动性的软件实体,逐渐演变为当前由LLM作为其“认知核心”的认知实体。我们将分析这种概念上的深刻

## Plan Draft & Refinement Artifacts

These artifacts capture the parsed plan fields after the draft and refinement steps in each round.
They are saved as JSON in `artifacts/` with filenames like `*_plan_run_*_plan_draft_round*.json` and `*_plan_run_*_plan_refine_round*.json`.


In [ ]:
display_prompt("""
System prompt used by the LLM:
```text
You are a research planning agent. Produce a plan that is concise, actionable, and focused on gaps. All natural-language content MUST be in the requested output language.
Return ONE valid JSON object only. No markdown, no commentary, no extra text.

Requirements:
- Must include keys: scope, key_questions, keywords, gaps, notes, readiness
- scope, key_questions, keywords MUST be non-empty arrays (>= 3 items each)
- keywords must include core terms from the topic and interests; include extracted interests if provided
- methods are analysis approaches; use them to frame the plan, not as the topic itself
- gaps and notes may be empty arrays
- retrieval_queries is optional; if provided, it must be a short list of search phrases
- readiness must be one of: "draft", "refined", "ready"
- Output must be strict JSON (double quotes, no trailing commas)
- All natural-language content must be in the requested output language.
```
""")


In [6]:
plan_prefix = _pick_plan_prefix(RUN_PREFIX)
print(f"PLAN_PREFIX: {plan_prefix}")

drafts = []
refines = []
if plan_prefix:
    drafts = sorted(ARTIFACTS_DIR.glob(plan_prefix + "_plan_draft_round*.json"))
    refines = sorted(ARTIFACTS_DIR.glob(plan_prefix + "_plan_refine_round*.json"))
else:
    drafts = sorted(ARTIFACTS_DIR.glob("*_plan_draft_round*.json"), key=lambda p: p.stat().st_mtime)
    refines = sorted(ARTIFACTS_DIR.glob("*_plan_refine_round*.json"), key=lambda p: p.stat().st_mtime)
    if AUTO_PICK_LATEST:
        drafts = drafts[-1:]
        refines = refines[-1:]

_print_plan_group(drafts, "Plan Draft Artifacts")
_print_plan_group(refines, "Plan Refinement Artifacts")


PLAN_PREFIX: 2026_02_25_22_59_27_plan_run_5
=== Plan Draft Artifacts ===
(not found)
=== Plan Refinement Artifacts ===
=== 2026_02_25_22_59_27_plan_run_5_plan_refine_round2.json ===
File: artifacts/2026_02_25_22_59_27_plan_run_5_plan_refine_round2.json
{
  "label": "plan_refine_round2",
  "topic": "AI agent-driven workflows",
  "created_at": "2026-02-26T03:58:55.745177+00:00",
  "round_number": 2,
  "readiness": "refined",
  "scope": [
    "系统性评估:专注于开发和应用标准化基准,以比较不同AI智能体框架(如CrewAI, LangChain)在可靠性,成本效益和可扩展性方面的表现,特别是在执行复杂,多步骤工作流时。",
    "人机协作模型:研究将AI智能体集成到专家工作流(如科学研究,医疗诊断)中的交互模式,控制机制和信任建立策略,重点关注人机在环(Human-in-the-loop)的设计。",
    "多智能体动力学与经济性:分析多智能体系统(Multi-Agent Systems)的协调策略,涌现行为及其经济模型,特别是在API调用成本和计算资源分配方面的优化问题。"
  ],
  "key_questions": [
    "在AI驱动的科学发现(Agentic Science)工作流中,如何设计有效的人机协作(Human-in-the-loop)机制来引导智能体,验证中间结果并建立领域专家的信任？",
    "对于多智能体协作框架(如CrewAI, AstroAgents),如何量化评估其通信效率,任务分配鲁棒性和协作解决复杂问题的“涌现”能力？是否存在标准化的评估基准？",
    "随着AI智能体工作流的复杂性增加,其对大型语言模型(LLM)API和外部工具的调用成本如何扩展？有哪些策略可以在保证任务性能

## Plan Artifact

Input is the finalized plan fields and the selected outline. The plan is triggered at the end of the research run and rendered into markdown by `render_plan_md()` after the LLM produces plan fields. Plan generation happens before outline building and refinement rounds if needed.


In [ ]:
display_prompt("""
System prompt used by the LLM:
```text
You are a research planning agent. Produce a plan that is concise, actionable, and focused on gaps. All natural-language content MUST be in the requested output language.
Return ONE valid JSON object only. No markdown, no commentary, no extra text.

Requirements:
- Must include keys: scope, key_questions, keywords, gaps, notes, readiness
- scope, key_questions, keywords MUST be non-empty arrays (>= 3 items each)
- keywords must include core terms from the topic and interests; include extracted interests if provided
- methods are analysis approaches; use them to frame the plan, not as the topic itself
- gaps and notes may be empty arrays
- retrieval_queries is optional; if provided, it must be a short list of search phrases
- readiness must be one of: "draft", "refined", "ready"
- Output must be strict JSON (double quotes, no trailing commas)
- All natural-language content MUST be in the requested output language, except retrieval_queries must be in English.
- Keep JSON keys in English.
- Keep paper titles, dataset names, benchmarks, model names, APIs, and acronyms in English.
- Use ASCII punctuation for JSON (":" and ","), not full-width punctuation.
- When evidence includes confidence or recency, prefer higher-confidence and more recent evidence.
All natural-language content MUST be in the requested output language.

Example (Chinese):
{"scope":["..."],"key_questions":["..."],"keywords":["..."],"gaps":["..."],"notes":["..."],"readiness":"draft","retrieval_queries":["..."]}

Example (English):
{"scope":["..."],"key_questions":["..."],"keywords":["..."],"gaps":["..."],"notes":["..."],"readiness":"draft","retrieval_queries":["..."]}
```

User prompt template sent to the LLM:
```json
{
  "topic": "<topic>",
  "interests": ["<interest>", "..."],
  "methods": ["<method>", "..."],
  "documents": [
    {"title": "<title>", "source_type": "<type>", "source": "<source>", "doc_id": "<id>"}
  ],
  "output_language": "<language>",
  "instructions": {
    "output_json_schema": {
      "scope": ["string"],
      "key_questions": ["string"],
      "keywords": ["string"],
      "gaps": ["string"],
      "notes": ["string"],
      "retrieval_queries": ["string"],
      "readiness": "draft | refined | ready"
    },
    "language_policy": [
      "All list items must be in <language>.",
      "retrieval_queries must be in English.",
      "Keep paper titles, dataset names, benchmarks, model names, APIs, and acronyms in English."
    ],
    "keyword_guidance": [
      "Include key terms from the topic.",
      "Include salient terms from manually added interests.",
      "If extracted_interests are provided, incorporate them.",
      "Do not replace the topic with method names."
    ],
    "retrieval_query_guidance": [
      "If provided, use 2-6 word search phrases.",
      "Avoid negations like 'no' or 'lack of'.",
      "Focus on entities, relationships, and concrete nouns.",
      "Keep to 2-8 total queries.",
      "Queries must be in English."
    ],
    "method_guidance": [
      "Methods are analysis approaches (lenses), not standalone topics.",
      "Apply methods to the topic and evidence."
    ],
    "kg_guidance": [
      "If kg_fact_cards is provided, use it to ground claims and plan steps.",
      "Prefer higher-confidence and more recent evidence when shaping the plan.",
      "Do not copy evidence snippets verbatim; paraphrase.",
      "Do not fabricate sources that are not in kg_fact_cards or documents."
    ]
  },
  "extracted_interests": ["<extracted>", "..."],
  "kg_fact_cards": ["<fact card>", "..."],
  "skill_guidance": ["<skill guidance>"]
}
```
""")


In [7]:
plan = None
plan_prefix = None
if RUN_TIMESTAMP:
    plan_prefix = RUN_TIMESTAMP[:16]  # YYYY_MM_DD_HH_MM
elif RUN_PREFIX:
    match = re.match(r"^(\d{4}_\d{2}_\d{2}_\d{2}_\d{2}_\d{2})", RUN_PREFIX)
    if match:
        plan_prefix = match.group(1)[:16]

if plan_prefix:
    candidates = list(ARTIFACTS_DIR.glob(plan_prefix + "_plan.md"))
    if candidates:
        plan = max(candidates, key=lambda p: p.stat().st_mtime)
if not plan:
    candidates = list(ARTIFACTS_DIR.glob("*_plan.md"))
    if candidates:
        plan = max(candidates, key=lambda p: p.stat().st_mtime)

_print_markdown(plan, "Plan Artifact")


=== Plan Artifact ===
File: artifacts/2026_02_26_04_33_plan.md
# Research Plan - Technology analysis of AI agent-driven workflows

- Created at: 2026-02-26T09:26:13.370653+00:00
- Round: 2
- Readiness: refined

## Scope
- 深入分析构成AI智能体工作流的核心技术模块,包括规划(Planning),记忆(Memory),工具使用(Tool Use)和行动(Action)的具体实现机制。
- 对比分析主流智能体架构(如基于ReAct的单智能体,基于CrewAI或AutoGen的多智能体协作框架)在任务分解,执行效率和鲁棒性方面的差异。
- 重点研究“代理科学”(Agentic Science)范式下,AI智能体工作流在自主科学发现等前沿领域的具体应用案例,技术挑战和未来趋势。

## Key Questions
- 在不同的AI智能体架构中,规划,记忆和工具使用等核心模块是如何协同工作的？其设计选择对智能体在处理复杂任务时的自主性和可靠性有何影响？
- 多智能体系统(如CrewAI)与单智能体工作流(如AutoGPT)相比,在解决复杂问题时,其协作和通信机制带来了哪些优势,又引入了哪些新的技术挑战(如任务分配,一致性维护)？
- 在“代理科学”等应用中,如何有效缓解或验证由大语言模型(LLM)“幻觉”引发的错误,以确保AI智能体在进行长期,复杂的科学研究工作流时输出结果的可靠性和可复现性？
- 当前评估AI智能体工作流性能的主要方法和基准(Benchmarks)是什么？它们在多大程度上能够衡量智能体在真实世界复杂场景中的泛化能力和效率？

## Keywords
- AI智能体, 工作流自动化, 多智能体系统, 大语言模型, 科学智能, 智能体架构, 代理科学, ReAct, RAG, 工具使用

## Source Types in Selected Docs
- web
- file

## Source Types in Library
- file: 12
- web: 99

## Gaps and Retrieval Needs
- 缺乏标准